In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('datas/201809-citibike-tripdata.csv')

In [12]:
df.head()

,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,1635,2018-09-01 00:00:05.2690,2018-09-01 00:27:20.6340,252.0,MacDougal St & Washington Sq,40.732264,-73.998522,366.0,Clinton Ave & Myrtle Ave,40.693261,-73.968896,25577,Subscriber,1980,1
1,132,2018-09-01 00:00:11.2810,2018-09-01 00:02:23.4810,314.0,Cadman Plaza West & Montague St,40.693830,-73.990539,3242.0,Schermerhorn St & Court St,40.691029,-73.991834,34377,Subscriber,1969,0
2,3337,2018-09-01 00:00:20.6490,2018-09-01 00:55:58.5470,3142.0,1 Ave & E 62 St,40.761227,-73.960940,3384.0,Smith St & 3 St,40.678724,-73.995991,30496,Subscriber,1975,1
3,436,2018-09-01 00:00:21.7460,2018-09-01 00:07:38.5830,308.0,St James Pl & Oliver St,40.713079,-73.998512,3690.0,Park Pl & Church St,40.713342,-74.009355,28866,Subscriber,1984,2
4,8457,2018-09-01 00:00:27.3150,2018-09-01 02:21:25.3080,345.0,W 13 St & 6 Ave,40.736494,-73.997044,380.0,W 4 St & 7 Ave S,40.734011,-74.002939,20943,Customer,1994,1


### 1. Найти общее количество строк и столбцов в датасете - указать первым число строк, вторым - число столбцов

In [13]:
df.shape

(1877884, 15)

### 2. Найти среднюю длину поездок в минутах(столбец tripduration) c точностью до 2 знака

In [14]:
round(df['tripduration'].mean()/60,2)

16.13

### 3. Сколько поездок начались и закончились в той же самой станции?

In [27]:
df[df['start station id']==df['end station id']].count()['start station id']

41364

### 4. Какой самый используемый байк(bikeid) в городе по количеству поездок?

In [17]:
bike_df = df['bikeid'] # создаем фрейм номеров байков со всех поездок
bike_frequency = bike_df.value_counts() # создаем фрейм, содержащий номер байка и кол-во поездок на нем
most_popular_bikeid = bike_frequency.index[0] # среди отсортированного по убыванию массивов номеров байков выбираем первый
print(most_popular_bikeid)

# print(df['bikeid'].value_counts().index[0]) # либо все вместе

33875


### 5. Найдите идентификатор велосипеда (bikeid), у которого в среднем продолжительность поездок выше, чем у всех остальных

In [59]:
bike_trip_df = df[['bikeid','tripduration']] # отдельный датафрейм, содержащ. только номер байка и продолж. поездки
bike_frequency = df['bikeid'].value_counts() # массив, содерж инфу о кол-ве поездок на каждом велосипеде
bike_frequency_df = pd.DataFrame({'bikeid':bike_frequency.index,'tripcounter':bike_frequency.values }) # создаем фрейм, содержащий номер байка и кол-во поездок на нем

max_average_duration = 0
most_bike = 0
for bike in bike_frequency_df['bikeid']: # рассматриваем каждый отдельный байк   
    bike_time_mean = bike_trip_df[bike_trip_df['bikeid']==bike]['tripduration'].mean() # средняя продолжит. поездки на опред. байке
    if max_average_duration < bike_time_mean: # сравниваем полученное ср. значение с предыдущим самым большим знач.
        max_average_duration, most_bike = bike_time_mean, bike # присваиваем most_bike значение байк, с самой высок. средней продолж. поездки
print(most_bike)

17548


### 6. Сколько строк, в которых отсутствуют данные о start station id?

In [33]:
df['start station id'].isna().sum() # кол-во пустых значений указанного столбца 

716

### 7. Какова средняя продолжительность поездки в минутах в зависимости от типа подписки c точностью до 2 знака?

In [71]:
average_duration_subscription = {} # словарь, сост. из всех типов подписок и ср. продолжительности поездки
for subscription in df['usertype'].unique(): # перебор всех типов подписок
    average_duration_subscription[subscription] = round(df[df['usertype']==subscription]['tripduration'].mean()/60,2)
print(average_duration_subscription)

{'Subscriber': 13.33, 'Customer': 33.42}


### 8. Найдите среднюю длину поездок в километрах с точностью до 2 знака, предварительно выкинув замкнутые траектории(те у которых совпадают start station id = end station id).

##### Hint: можно воспользоваться библиотекой geopy и взять расстояние vincenty(минимальное расстояние между точками)

In [13]:
from geopy.distance import geodesic
coordinates_df = df[df['start station id']!=df['end station id']][['start station latitude','start station longitude','end station latitude','end station longitude']] # отсеяли замкнутые траектории, взяли только некоторые столбцы

dist_list = [geodesic(coordinates_df.values[i][0:2], coordinates_df.values[i][2:4]).kilometers for i in range(len(coordinates_df))]
print(np.mean(dist_list))

1.8495860872213767


### 9. Выберите станцию (start station id) с максимальным количеством отправлений с 18 до 20 вечера включительно

In [88]:
time_df = df[['starttime','start station id']] # отдельный дф с 2мя столбцами
time_df['starthour'] = [d.hour for d in pd.to_datetime(time_df['starttime'], yearfirst = True)] # новый столбец со знач. часа начала проката
time_df = time_df[ time_df['starthour'].isin(range(18,21))] # берем только значения с началом отправления в данном промежутке
print(int(time_df['start station id'].value_counts().index[0])) # выводим самое часто встречаемое число

519


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


### 10. Выберите идентификаторы станций(end station id), в которые приезжают с 6 до 10 утра включительно

In [101]:
time_df = df[['stoptime','end station id']] # отдельный дф с 2мя столбцами
time_df['finishhour'] = [d.hour for d in pd.to_datetime(time_df['stoptime'], yearfirst = True)] # новый столбец со знач. часа окончания проката
time_df = time_df[ time_df['finishhour'].isin(range(6,10))] # берем только значения с окончанием отправления в данном промежутке
[x in time_df['end station id'].unique() for x in [3140,3106,3116,369]] # прочерка номеров байков на вхождение в дф

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


[True, True, True, True]